# Testing SWOT processing

In [11]:
# import pysatview

### SWOT processing tests

Test the login

In [2]:
# from pysatview.swot.swot_processor import SwotDataProcessor

# swotpy = SwotDataProcessor()
# swotpy.authenticate()

In [1]:
# from pysatview.swot import swot_processor
# swot_processor = swot_processor
# swot_processor.SwotDataProcessor()
# swot_processor.create_file_monitor()

Test the project workflow

In [1]:
mnf_inner = {
    "west_lon": 113.0,
    "east_lon": 114.0,
    "south_lat": -23.0,
    "north_lat": -21.0
}
smallbox=([mnf_inner['west_lon'], mnf_inner['east_lon']],
          [mnf_inner['south_lat'], mnf_inner['north_lat']])

In [3]:
from pysatview.swot.swot_processor import SwotWorkflow

# Configuration parameters
# timelims = ("2025-10-01T00:00:00", datetime.utcnow().isoformat())
timelims = ("2026-03-05T00:00:00Z", "2026-03-10T00:00:00Z")
lonlims = (111, 114)  # Western Australia region
latlims = (-25, -20)

# sname = "SWOT_L2_LR_SSH_Expert_D"
sname = "SWOT_L2_LR_PreCalSSH_D"
data_types = {'Expert':['ssha_karin', 'swh_karin', 'sig0_karin']}

# Create workflow instance
workflow = SwotWorkflow()

# Run complete workflow
workflow.run_complete_workflow(short_name=sname, data_type=data_types, timelims=timelims, lonlims=lonlims, latlims=latlims, smallbox=smallbox, only_last=True)

=== SWOT Data Processing Workflow ===

1. Setting up authentication...
✓ Successfully authenticated with Earthdata
2026-02-27 06:36:26.114059
🔍 Searching SWOT data... 2026-02-27T07:06:26 to 2026-03-10T00:00:00Z
Found 20 granules
📥 Downloading...


QUEUEING TASKS | : 100%|██████████| 4/4 [00:00<00:00, 1796.85it/s]
PROCESSING TASKS | :  25%|██▌       | 1/4 [02:09<06:29, 129.83s/it]


KeyboardInterrupt: 

In [8]:
from pathlib import Path
import shutil
from datetime import datetime, timezone
from PIL import Image

base_dir = workflow.processor.base_dir

def latest_pdf(png_dirs):
    '''Move files and create the latest PDF'''
    # Make the latest dir
    latest_dir = Path(base_dir.parent / 'latest')
    latest_dir.mkdir(parents=True, exist_ok=True)
    
    for old_latest in latest_dir.iterdir():
        if old_latest.is_file():
            old_latest.unlink()
    
    # Add lestest PNGs to a combined PDF
    print("📄 Compiling latest PNGs into a PDF report...")
    pdf_files = []
    for png_dir in png_dirs:
        png_path = Path(png_dir)
        if png_path.exists() and png_path.is_dir():
            outer = [p for p in png_path.glob('*.png') if 'inner' not in p.name]
            if outer:
                pdf_files.append(max(outer, key=lambda p: p.stat().st_mtime))

            inner = [p for p in png_path.glob('*.png') if 'inner' in p.name]
            if inner:
                pdf_files.append(max(inner, key=lambda p: p.stat().st_mtime))
                
    # Copy all latest pngs to latest folder
    for png in pdf_files:
        shutil.copy2(png, latest_dir / png.name)
        
    # Create PDF
    imgs = [Image.open(p) for p in pdf_files]
    imgs = [im.convert("RGB") for im in imgs]

    # first image .save() with the rest as an append list
    out_path = latest_dir / (datetime.now(timezone.utc).strftime("%Y%m%d_%H") + "h_latest.pdf")
    imgs[0].save(out_path, save_all=True, append_images=imgs[1:])
    print(f"Created {out_path.name} with {len(imgs)} pages")  

In [9]:
latest_pdf([workflow.processor.png_dir])

📄 Compiling latest PNGs into a PDF report...
Created 20260310_14h_latest.pdf with 2 pages


Delete on of the files and one of the figures and run the file monitor

In [2]:
# from pysatview.himawari.himawari_processor import HimawariFileMonitor

# monitor_results = HimawariFileMonitor(processor=workflow.processor).check_file_completeness(timelims=timelims)
# print(monitor_results)

# HimawariFileMonitor(processor=workflow.processor).repair_missing_files(monitor_results, lonlims=lonlims, latlims=latlims)